In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from itertools import combinations
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#Dataset
torch.random.manual_seed(42)
unlabeled = 30
num_classes = 2
label_budget = 5

data = torch.empty(unlabeled, num_classes).uniform_(-1, 1)
labels = (data[:, 1] >= data[:, 0]).float().unsqueeze(1)
#print("Data shape:", data.shape)
#print("Labels shape:", labels.shape)

all_subsets = list(combinations(range(unlabeled), label_budget))
x_all = torch.stack([data[list(subset)] for subset in all_subsets])
y_all = torch.stack([labels[list(subset)] for subset in all_subsets])
#print("x_all shape:", x_all.shape)
#print("y_all shape:", y_all.shape)


In [4]:

pt_idx = [2, 3, 4, 5]
target_set = set(pt_idx)

found = False
for i, subset in enumerate(all_subsets):
    if set(subset) == target_set:
        print(f"Subset found!")
        print(f"Index: {i}")
        print(f"Points: {subset}")
        print(f"Labels: {y_all[i].flatten().tolist()}")
        found = True
        break 

if not found:
    print("Subset not found.")

Subset found!
Index: 1785
Points: (2, 3, 4, 5)
Labels: [1.0, 1.0, 0.0, 0.0]


In [ ]:
# x @ W.T + b
N = len(all_subsets) #number of subsets of size 5 from 30 samples = C(30, 5) = 142506
print(N)
W = torch.randn(N, 1, 2, requires_grad=True)
b = torch.randn(N, 1, requires_grad=True)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD([W, b], lr=0.1)


4845


In [6]:
# Training Loop
losses = []
for epoch in range(100):
    optimizer.zero_grad()
    logits = x_all @ W.transpose(-1, -2) + b.unsqueeze(1)
    loss = criterion(logits, y_all)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print(f'Final Loss: {loss.item():.4f}')

Final Loss: 0.8625


In [7]:
#test dataset
test_data = torch.cartesian_prod(torch.linspace(-1, 1, 20), torch.linspace(-1, 1, 20))
test_labels = (test_data[:, 1] >= test_data[:, 0]).float().unsqueeze(1)
test_dataset = TensorDataset(test_data, test_labels)
test_loader = DataLoader(test_dataset, batch_size=400, shuffle=False)

In [8]:
# Evaluate test dataset
with torch.no_grad():
    for x, y in test_loader:
        logits = x.unsqueeze(0) @ W.transpose(-1, -2) + b.unsqueeze(1)
        preds = (logits >= 0).float()
        accuracies = (preds == y.unsqueeze(0)).float().mean(dim=[1, 2])

#print(accuracies)
print(f'Random Sampling baseline: {accuracies.mean():.4f}')
#print(f'best subset accuracy: {accuracies.max():.4f}')
#print(f'worst subset accuracy: {accuracies.min():.4f}')
print(x.shape)



Random Sampling baseline: 0.5020
torch.Size([400, 2])


In [9]:
#print(accuracies)
#print(accuracies.shape)
#print(f'Subset {torch.max(accuracies, dim=0)[1]} has Max Accuracy: {torch.max(accuracies)}')
#print(f'Point Indices of Max Accuracy: {all_subsets[torch.max(accuracies, dim=0)[1]]} and the corrsponding labels are {(y_all[torch.max(accuracies, dim=0)[1]]).tolist()}')
#print(f'Subset {torch.min(accuracies, dim=0)[1]} has Min Accuracy: {torch.min(accuracies)}')
#print(f'Point Indices of Min Accuracy: {all_subsets[torch.min(accuracies, dim=0)[1]]} and the corrsponding labels are {(y_all[torch.min(accuracies, dim=0)[1]]).tolist()}')
#val = accuracies[4582]
#print(val)

In [10]:
'''nn_index = 4582 # jedes NN wird mit einem Subset trainiert, nn_index ist der Index des Subsets.
print(all_subsets[nn_index])
subset_indices = all_subsets[nn_index]
subset_labels = labels[list(subset_indices)]
print(f"Indices of Subset {nn_index} : {subset_indices}")
print(f"Labels of these 4 points:\n{subset_labels.flatten()}")
is_same_side = torch.all(subset_labels == subset_labels[0]).item()
print(f"Are these 4 points on the same side? {is_same_side}")
print(f"Accuracy for NN index {nn_index}: {accuracies[nn_index].item():.4f}")'''

'nn_index = 4582 # jedes NN wird mit einem Subset trainiert, nn_index ist der Index des Subsets.\nprint(all_subsets[nn_index])\nsubset_indices = all_subsets[nn_index]\nsubset_labels = labels[list(subset_indices)]\nprint(f"Indices of Subset {nn_index} : {subset_indices}")\nprint(f"Labels of these 4 points:\n{subset_labels.flatten()}")\nis_same_side = torch.all(subset_labels == subset_labels[0]).item()\nprint(f"Are these 4 points on the same side? {is_same_side}")\nprint(f"Accuracy for NN index {nn_index}: {accuracies[nn_index].item():.4f}")'

In [11]:
'''# visualization of a single subset's distribution
subset_indices = all_subsets[nn_index]
subset_points = data[list(subset_indices)].numpy()
subset_labels = labels[list(subset_indices)].flatten().numpy()

plt.figure(figsize=(6, 6))
plt.plot([-1, 1], [-1, 1], color='gray', linestyle='--', label='Ideal Boundary ($x_1=x_0$)')
plt.arrow(0, 0, -0.3, 0.3, head_width=0.05, color='gray', alpha=0.5, label='Ideal Direction')
# draw learned direction
# w0 = W[918, 0, 0], w1 = W[918, 0, 1] subset 918
w0 = W[nn_index, 0, 0].detach().cpu().item()
w1 = W[nn_index, 0, 1].detach().cpu().item()

# learning direction 
#plt.arrow(0, 0, w0, w1, head_width=0.05, color='blue', label='Learned Direction')
scale = 0.5 / (abs(w0) + abs(w1)) 
plt.arrow(0, 0, w0*scale, w1*scale, head_width=0.05, color='blue', label=f'Learned Direction ({nn_index})')

# draw 4 points
for i in range(4):
    color = 'green' if subset_labels[i] == 1 else 'red'
    marker = 'o' if subset_labels[i] == 1 else 'x'
    plt.scatter(subset_points[i, 0], subset_points[i, 1], c=color, marker=marker, s=100)

plt.xlim(-1.1, 1.1)
plt.ylim(-1.1, 1.1)
plt.xlabel('$x_0$')
plt.ylabel('$x_1$')
plt.title(f'Training Points in Subset {nn_index} (Label Distribution)')
plt.grid(True, alpha=0.3)
plt.legend()

plt.savefig(f'subset_{nn_index}_analysisd.png')
print(f"subset {nn_index} distribution analysis saved.")'''

'# visualization of a single subset\'s distribution\nsubset_indices = all_subsets[nn_index]\nsubset_points = data[list(subset_indices)].numpy()\nsubset_labels = labels[list(subset_indices)].flatten().numpy()\n\nplt.figure(figsize=(6, 6))\nplt.plot([-1, 1], [-1, 1], color=\'gray\', linestyle=\'--\', label=\'Ideal Boundary ($x_1=x_0$)\')\nplt.arrow(0, 0, -0.3, 0.3, head_width=0.05, color=\'gray\', alpha=0.5, label=\'Ideal Direction\')\n# draw learned direction\n# w0 = W[918, 0, 0], w1 = W[918, 0, 1] subset 918\nw0 = W[nn_index, 0, 0].detach().cpu().item()\nw1 = W[nn_index, 0, 1].detach().cpu().item()\n\n# learning direction \n#plt.arrow(0, 0, w0, w1, head_width=0.05, color=\'blue\', label=\'Learned Direction\')\nscale = 0.5 / (abs(w0) + abs(w1)) \nplt.arrow(0, 0, w0*scale, w1*scale, head_width=0.05, color=\'blue\', label=f\'Learned Direction ({nn_index})\')\n\n# draw 4 points\nfor i in range(4):\n    color = \'green\' if subset_labels[i] == 1 else \'red\'\n    marker = \'o\' if subset_la

In [12]:
'''# Visualization of test dataset classification results for a specific NN index
nn_index = 1785 # max:4582 min:3160 RF max: 2288 min: 958
preds_nn_index = preds[nn_index].squeeze()
correct = (preds_nn_index == y.squeeze()).numpy()
x_np = x.numpy()

fig, (ax_loss, ax_scatter) = plt.subplots(1, 2, figsize=(12, 5))

ax_loss.plot(losses)
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Loss')
ax_loss.set_title('Training Losses')

ax_scatter.scatter(x_np[correct, 0], x_np[correct, 1], c='green', marker='o', label='Correct')
ax_scatter.scatter(x_np[~correct, 0], x_np[~correct, 1], c='red', marker='x', label='Incorrect')
ax_scatter.plot([-1, 1], [-1, 1], color='gray', linestyle='--', label='Decision Boundary (x0=x1)')
ax_scatter.set_xlabel('x0')
ax_scatter.set_ylabel('x1')
ax_scatter.set_title('testset classification results')
ax_scatter.legend()

plt.tight_layout()
plt.savefig(f'{nn_index} NN_result.png')
print(f'{nn_index} NN_result.png saved')'''

"# Visualization of test dataset classification results for a specific NN index\nnn_index = 1785 # max:4582 min:3160 RF max: 2288 min: 958\npreds_nn_index = preds[nn_index].squeeze()\ncorrect = (preds_nn_index == y.squeeze()).numpy()\nx_np = x.numpy()\n\nfig, (ax_loss, ax_scatter) = plt.subplots(1, 2, figsize=(12, 5))\n\nax_loss.plot(losses)\nax_loss.set_xlabel('Epoch')\nax_loss.set_ylabel('Loss')\nax_loss.set_title('Training Losses')\n\nax_scatter.scatter(x_np[correct, 0], x_np[correct, 1], c='green', marker='o', label='Correct')\nax_scatter.scatter(x_np[~correct, 0], x_np[~correct, 1], c='red', marker='x', label='Incorrect')\nax_scatter.plot([-1, 1], [-1, 1], color='gray', linestyle='--', label='Decision Boundary (x0=x1)')\nax_scatter.set_xlabel('x0')\nax_scatter.set_ylabel('x1')\nax_scatter.set_title('testset classification results')\nax_scatter.legend()\n\nplt.tight_layout()\nplt.savefig(f'{nn_index} NN_result.png')\nprint(f'{nn_index} NN_result.png saved')"

In [13]:
'''# Visualization of NN accuracy distribution across all subsets
plt.figure(figsize=(18, 12))

# get hist return values: n (count), bins (edges), patches (bar objects)
n, bins, patches = plt.hist(accuracies.numpy(), bins=50, range=(0, 1), edgecolor='black', color='steelblue', alpha=0.5)

# get number of subests for each accuracy bin
for i in range(len(patches)):
    count = n[i]
    if count > 0: # only show number for bars with count > 0
        # get bar center x and height y
        x = patches[i].get_x() + patches[i].get_width() / 2
        y = patches[i].get_height()
        
        # show count above the bar, with a small offset (max(n)*0.01) to avoid overlap with the bar
        plt.text(x, y + (max(n)*0.01), str(int(count)), 
                 ha='center', va='bottom', fontsize=12, rotation=60)
        
plt.axvline(np.mean(accuracies.numpy()), color='blue', linestyle='--', label=f'Mean: {np.mean(accuracies.numpy()):.4f}')
plt.xlabel('Accuracy')
plt.ylabel('Number of Subsets')
plt.title(f'Distribution of Test Accuracy across all {len(all_subsets)} (n={unlabeled}, k={label_budget})Subsets w/ Counts')
plt.legend(fontsize=24)
plt.tight_layout()
plt.savefig(f'Distribution {len(all_subsets)}, n={unlabeled}, k={label_budget} with Counts.png')'''

"# Visualization of NN accuracy distribution across all subsets\nplt.figure(figsize=(18, 12))\n\n# get hist return values: n (count), bins (edges), patches (bar objects)\nn, bins, patches = plt.hist(accuracies.numpy(), bins=50, range=(0, 1), edgecolor='black', color='steelblue', alpha=0.5)\n\n# get number of subests for each accuracy bin\nfor i in range(len(patches)):\n    count = n[i]\n    if count > 0: # only show number for bars with count > 0\n        # get bar center x and height y\n        x = patches[i].get_x() + patches[i].get_width() / 2\n        y = patches[i].get_height()\n\n        # show count above the bar, with a small offset (max(n)*0.01) to avoid overlap with the bar\n        plt.text(x, y + (max(n)*0.01), str(int(count)), \n                 ha='center', va='bottom', fontsize=12, rotation=60)\n\nplt.axvline(np.mean(accuracies.numpy()), color='blue', linestyle='--', label=f'Mean: {np.mean(accuracies.numpy()):.4f}')\nplt.xlabel('Accuracy')\nplt.ylabel('Number of Subsets'

In [14]:
'''# set target accuracy and tolerance
target_acc = 0.9
tolerance = 0.01

# find indices of subsets with accuracy close to 0.47
indices_near_047 = torch.where((accuracies >= target_acc - tolerance) & 
                               (accuracies <= target_acc + tolerance))[0]

print(f"Found {len(indices_near_047)} subsets with accuracy close to {target_acc} ± {tolerance}.")

# randomly select a few indices from the filtered list
num_samples = min(100, len(indices_near_047))
selected_indices = indices_near_047[torch.randperm(len(indices_near_047))[:num_samples]]'''

'# set target accuracy and tolerance\ntarget_acc = 0.9\ntolerance = 0.01\n\n# find indices of subsets with accuracy close to 0.47\nindices_near_047 = torch.where((accuracies >= target_acc - tolerance) & \n                               (accuracies <= target_acc + tolerance))[0]\n\nprint(f"Found {len(indices_near_047)} subsets with accuracy close to {target_acc} ± {tolerance}.")\n\n# randomly select a few indices from the filtered list\nnum_samples = min(100, len(indices_near_047))\nselected_indices = indices_near_047[torch.randperm(len(indices_near_047))[:num_samples]]'

In [15]:
'''import numpy as np
# get original data and labels as numpy arrays for plotting
data_np = data.cpu().numpy()
labels_np = labels.flatten().cpu().numpy()

# count how many times each point is selected in these 100 subsets
point_counts47 = np.zeros(len(data))
for idx in selected_indices:
    subset = all_subsets[idx]
    for p_idx in subset:
        point_counts47[p_idx] += 1

plt.figure(figsize=(8, 8))

# decision boundary
plt.plot([-1, 1], [-1, 1], color='gray', linestyle='--', alpha=0.5, label='Ideal Boundary')

# draw all 20 points, size based on how many times they were selected in the "mediocre subsets"
for i in range(len(data)):
    color = 'green' if labels_np[i] == 1 else 'blue'
    marker = 'o' if labels_np[i] == 1 else 'D'
    size = 20 + (point_counts47[i] * 20) 
    plt.scatter(data_np[i, 0], data_np[i, 1], c=color, marker=marker, s=size, edgecolors='k', alpha=0.6)

plt.title(f'Point Distribution of {num_samples} Subsets (Acc ≈ {target_acc})\nLarger points appear more frequently in these subsets')
plt.xlabel('x0')
plt.ylabel('x1')
plt.grid(True, alpha=0.2)
plt.savefig(f'Analysis_{target_acc}_Points.png')'''

'import numpy as np\n# get original data and labels as numpy arrays for plotting\ndata_np = data.cpu().numpy()\nlabels_np = labels.flatten().cpu().numpy()\n\n# count how many times each point is selected in these 100 subsets\npoint_counts47 = np.zeros(len(data))\nfor idx in selected_indices:\n    subset = all_subsets[idx]\n    for p_idx in subset:\n        point_counts47[p_idx] += 1\n\nplt.figure(figsize=(8, 8))\n\n# decision boundary\nplt.plot([-1, 1], [-1, 1], color=\'gray\', linestyle=\'--\', alpha=0.5, label=\'Ideal Boundary\')\n\n# draw all 20 points, size based on how many times they were selected in the "mediocre subsets"\nfor i in range(len(data)):\n    color = \'green\' if labels_np[i] == 1 else \'blue\'\n    marker = \'o\' if labels_np[i] == 1 else \'D\'\n    size = 20 + (point_counts47[i] * 20) \n    plt.scatter(data_np[i, 0], data_np[i, 1], c=color, marker=marker, s=size, edgecolors=\'k\', alpha=0.6)\n\nplt.title(f\'Point Distribution of {num_samples} Subsets (Acc ≈ {target

In [16]:
'''# bar chart for acc=0.47/0.30/0.90 in 3 colors (0-19 are indices of the unlabeled points)
plt.figure(figsize=(8, 8))
x = np.arange(20)
plt.bar(x - 0.2, point_counts47/point_counts47.sum(), width=0.2, label='acc≈0.47')
plt.bar(x,       point_counts30/point_counts30.sum(), width=0.2, label='acc≈0.30')
plt.bar(x + 0.2, point_counts90/point_counts90.sum(), width=0.2, label='acc≈0.90')
plt.xlabel('Point Index')
plt.ylabel('Normalized Frequency')
plt.title('Distribution of Selected Points Across Subsets')
plt.legend()
plt.savefig('histogram.png')'''

"# bar chart for acc=0.47/0.30/0.90 in 3 colors (0-19 are indices of the unlabeled points)\nplt.figure(figsize=(8, 8))\nx = np.arange(20)\nplt.bar(x - 0.2, point_counts47/point_counts47.sum(), width=0.2, label='acc≈0.47')\nplt.bar(x,       point_counts30/point_counts30.sum(), width=0.2, label='acc≈0.30')\nplt.bar(x + 0.2, point_counts90/point_counts90.sum(), width=0.2, label='acc≈0.90')\nplt.xlabel('Point Index')\nplt.ylabel('Normalized Frequency')\nplt.title('Distribution of Selected Points Across Subsets')\nplt.legend()\nplt.savefig('histogram.png')"

In [17]:
'''# Random Forest on toy dataset test for 3 subsets (best, worst)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Tensor to Numpy
x_test_np = test_data.cpu().numpy()
y_test_np = test_labels.flatten().cpu().numpy()

# 2. 3 subsets to test 
indices_to_test = [4582, 918, 3160]

print("--- Random Forest Evaluation ---")

for idx in indices_to_test:
    # get training data for these 3 subsets
    subset_idx = all_subsets[idx]
    x_train_rf = data[list(subset_idx)].cpu().numpy()
    y_train_rf = labels[list(subset_idx)].flatten().cpu().numpy()
    
    # Random Forest Classifier
    rf_model = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
    
    # Train the model
    rf_model.fit(x_train_rf, y_train_rf)
    
    # Predict and evaluate
    rf_preds = rf_model.predict(x_test_np)
    rf_acc = accuracy_score(y_test_np, rf_preds)
    
    print(f"Subset Index {idx}:")
    print(f"  - Neural Network Accuracy: {accuracies[idx]:.4f}")
    print(f"  - Random Forest Accuracy:  {rf_acc:.4f}")
    print("-" * 30)'''

'# Random Forest on toy dataset test for 3 subsets (best, worst)\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score\n\n# 1. Tensor to Numpy\nx_test_np = test_data.cpu().numpy()\ny_test_np = test_labels.flatten().cpu().numpy()\n\n# 2. 3 subsets to test \nindices_to_test = [4582, 918, 3160]\n\nprint("--- Random Forest Evaluation ---")\n\nfor idx in indices_to_test:\n    # get training data for these 3 subsets\n    subset_idx = all_subsets[idx]\n    x_train_rf = data[list(subset_idx)].cpu().numpy()\n    y_train_rf = labels[list(subset_idx)].flatten().cpu().numpy()\n\n    # Random Forest Classifier\n    rf_model = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)\n\n    # Train the model\n    rf_model.fit(x_train_rf, y_train_rf)\n\n    # Predict and evaluate\n    rf_preds = rf_model.predict(x_test_np)\n    rf_acc = accuracy_score(y_test_np, rf_preds)\n\n    print(f"Subset Index {idx}:")\n    print(f"  - Neural Network Accura

In [18]:
# Random Forest on toy dataset test for all subsets
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

accuracies_rf = []
count = 0
print(f"--- Random Forest Training and Evaluating for All {len(all_subsets)} Subsets ---")
for idx in range(len(all_subsets)):
    subset_idx = all_subsets[idx]
    x_train_rf = data[list(subset_idx)].cpu().numpy()
    y_train_rf = labels[list(subset_idx)].flatten().cpu().numpy()
    
    rf_model = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
    rf_model.fit(x_train_rf, y_train_rf)
    
    x_test_np = test_data.cpu().numpy()
    y_test_np = test_labels.flatten().cpu().numpy()

    rf_preds = rf_model.predict(x_test_np)
    rf_acc = accuracy_score(y_test_np, rf_preds)
    accuracies_rf.append(rf_acc)
    count += 1
    if count % 1000 == 0:
        print(f" {count} / {len(all_subsets)} completed")



--- Random Forest Training and Evaluating for All 4845 Subsets ---
 1000 / 4845 completed
 2000 / 4845 completed
 3000 / 4845 completed
 4000 / 4845 completed


In [19]:
'''# visualize RF accuracy distribution across all subsets
plt.figure(figsize=(18, 12))
n, bins, patches = plt.hist(accuracies_rf, bins=50, range=(0, 1), edgecolor='black', color='sandybrown', alpha=0.7)

# get the count for each bar and show it above the bar
for i in range(len(patches)):
    count = n[i]
    if count > 0:
        x_pos = patches[i].get_x() + patches[i].get_width() / 2
        y_pos = patches[i].get_height()
        plt.text(x_pos, y_pos + (max(n) * 0.01), str(int(count)), 
                 ha='center', va='bottom', fontsize=12, rotation=60)

plt.axvline(np.mean(accuracies_rf), color='blue', linestyle='--', label=f'Mean: {np.mean(accuracies_rf):.4f}')
plt.xlabel('Test Accuracy')
plt.ylabel('Number of Subsets')
plt.title(f'Random Forest Performance Distribution {len(all_subsets)} Subsets (n={unlabeled}, k={label_budget})')
plt.legend(fontsize=24)
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()

plt.savefig(f'RF_Accuracy_Distribution_n{unlabeled}_k{label_budget}.png')
plt.show()'''

"# visualize RF accuracy distribution across all subsets\nplt.figure(figsize=(18, 12))\nn, bins, patches = plt.hist(accuracies_rf, bins=50, range=(0, 1), edgecolor='black', color='sandybrown', alpha=0.7)\n\n# get the count for each bar and show it above the bar\nfor i in range(len(patches)):\n    count = n[i]\n    if count > 0:\n        x_pos = patches[i].get_x() + patches[i].get_width() / 2\n        y_pos = patches[i].get_height()\n        plt.text(x_pos, y_pos + (max(n) * 0.01), str(int(count)), \n                 ha='center', va='bottom', fontsize=12, rotation=60)\n\nplt.axvline(np.mean(accuracies_rf), color='blue', linestyle='--', label=f'Mean: {np.mean(accuracies_rf):.4f}')\nplt.xlabel('Test Accuracy')\nplt.ylabel('Number of Subsets')\nplt.title(f'Random Forest Performance Distribution {len(all_subsets)} Subsets (n={unlabeled}, k={label_budget})')\nplt.legend(fontsize=24)\nplt.grid(axis='y', alpha=0.2)\nplt.tight_layout()\n\nplt.savefig(f'RF_Accuracy_Distribution_n{unlabeled}_k{la

In [20]:
'''print(f'NN  mean accuracy: {accuracies.mean():.4f}')
print(f'RF  mean accuracy: {np.mean(accuracies_rf):.4f}')
print(f'NN  max accuracy: {accuracies.max():.4f}')
print(f'RF  max accuracy: {np.max(accuracies_rf):.4f}')
print(f'NN  min accuracy: {accuracies.min():.4f}')
print(f'RF  min accuracy: {np.min(accuracies_rf):.4f}')'''


"print(f'NN  mean accuracy: {accuracies.mean():.4f}')\nprint(f'RF  mean accuracy: {np.mean(accuracies_rf):.4f}')\nprint(f'NN  max accuracy: {accuracies.max():.4f}')\nprint(f'RF  max accuracy: {np.max(accuracies_rf):.4f}')\nprint(f'NN  min accuracy: {accuracies.min():.4f}')\nprint(f'RF  min accuracy: {np.min(accuracies_rf):.4f}')"

In [21]:
fig, ax = plt.subplots(figsize=(10, 5))
common_range = (0, 1)
common_bins = 50
ax.hist(accuracies.numpy(), bins=common_bins, range=common_range, alpha=0.5, edgecolor='black', label='Neural Network')
ax.hist(accuracies_rf,      bins=common_bins, range=common_range, alpha=0.5, edgecolor='black', label='Random Forest')
ax.set_xlabel('Accuracy')
ax.set_ylabel('Number of Subsets')
ax.set_title(f'NN vs Random Forest: Distribution of Test Accuracy {len(all_subsets)} (n={unlabeled}, k={label_budget})')
ax.legend()

table_data = [
    [f'{accuracies.max():.4f}',  f'{np.max(accuracies_rf):.4f}'],
    [f'{accuracies.mean():.4f}', f'{np.mean(accuracies_rf):.4f}'],
    [f'{accuracies.min():.4f}',  f'{np.min(accuracies_rf):.4f}'],
]

table = ax.table(
    cellText=table_data,
    rowLabels=['Max', 'Mean', 'Min'],
    colLabels=['NN', 'RF'],
    loc='top left',        
    cellLoc='center',        
    bbox=[0.06, 0.78, 0.2, 0.2]
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(0.3, 0.6) 

plt.tight_layout()
plt.savefig(f'NN_vs_RF_distribution(n={unlabeled}, k={label_budget}, {len(all_subsets)}).png')

In [22]:
distances = (data[:, 1] - data[:, 0]).abs() / (2 ** 0.5)  # distance from the decision boundary (x=y)

subset_mean_distances = []
for subset in all_subsets:
    d = distances[list(subset)].mean().item()
    subset_mean_distances.append(d)

subset_mean_distances = np.array(subset_mean_distances)
correlation = np.corrcoef(subset_mean_distances, accuracies.numpy())[0, 1]
print(f'Correlation: {correlation:.4f}')

plt.figure(figsize=(10, 8))
plt.scatter(subset_mean_distances, accuracies.numpy(), alpha=0.1)
plt.title(f'Accuracy vs Mean Distance to Decision Boundary (n={unlabeled}, k={label_budget}), cor={correlation:.4f}')
plt.xlabel('Mean Distance to Decision Boundary')
plt.ylabel('Accuracy')
plt.savefig(f'Accuracy_vs_MeanDistance(n={unlabeled}, k={label_budget}).png')


Correlation: 0.0087


In [23]:
print(f'Subset {torch.max(accuracies, dim=0)[1]} has Max Accuracy by NN: {torch.max(accuracies)}')
print(f'Point Indices of Max Accuracy: {all_subsets[torch.max(accuracies, dim=0)[1]]} and the corrsponding labels are {(y_all[torch.max(accuracies, dim=0)[1]]).tolist()}')
print(f'Subset {torch.min(accuracies, dim=0)[1]} has Min Accuracy by NN: {torch.min(accuracies)}')
print(f'Point Indices of Min Accuracy: {all_subsets[torch.min(accuracies, dim=0)[1]]} and the corrsponding labels are {(y_all[torch.min(accuracies, dim=0)[1]]).tolist()}')
print(f'Subset {np.argmax(accuracies_rf)} has Max Accuracy by RF: {np.max(accuracies_rf):.4f}')
print(f'Point Indices of Max Accuracy: {all_subsets[np.argmax(accuracies_rf)]} and the corrsponding labels are {(y_all[np.argmax(accuracies_rf)]).tolist()}')
print(f'Subset {np.argmin(accuracies_rf)} has Min Accuracy by RF: {np.min(accuracies_rf):.4f}')
print(f'Point Indices of Min Accuracy: {all_subsets[np.argmin(accuracies_rf)]} and the corrsponding labels are {(y_all[np.argmin(accuracies_rf)]).tolist()}')

print(f'Subsets 0: {all_subsets[0]} with labels {y_all[0].flatten().tolist()} has mean distance {subset_mean_distances[0]:.4f} and accuracy {accuracies[0].item():.4f} by NN and {accuracies_rf[0]:.4f} by RF')
print(f'Subsets 4582: {all_subsets[4582]} with labels {y_all[4582].flatten().tolist()} has mean distance {subset_mean_distances[4582]:.4f} and accuracy {accuracies[4582].item():.4f} by NN and {accuracies_rf[4582]:.4f} by RF')
print(f'Subsets 3160: {all_subsets[3160]} with labels {y_all[3160].flatten().tolist()} has mean distance {subset_mean_distances[3160]:.4f} and accuracy {accuracies[3160].item():.4f} by NN and {accuracies_rf[3160]:.4f} by RF')
print(f'Subsets 2288: {all_subsets[2288]} with labels {y_all[2288].flatten().tolist()} has mean distance {subset_mean_distances[2288]:.4f} and accuracy {accuracies[2288].item():.4f} by NN and {accuracies_rf[2288]:.4f} by RF')
print(f'Subsets 958: {all_subsets[958]} with labels {y_all[958].flatten().tolist()} has mean distance {subset_mean_distances[958]:.4f} and accuracy {accuracies[958].item():.4f} by NN and {accuracies_rf[958]:.4f} by RF')

Subset 4582 has Max Accuracy by NN: 1.0
Point Indices of Max Accuracy: (9, 12, 13, 17) and the corrsponding labels are [[1.0], [1.0], [0.0], [1.0]]
Subset 3160 has Min Accuracy by NN: 0.0
Point Indices of Min Accuracy: (4, 6, 11, 14) and the corrsponding labels are [[0.0], [0.0], [1.0], [0.0]]
Subset 2288 has Max Accuracy by RF: 0.8850
Point Indices of Max Accuracy: (2, 8, 14, 18) and the corrsponding labels are [[1.0], [0.0], [0.0], [1.0]]
Subset 958 has Min Accuracy by RF: 0.3175
Point Indices of Min Accuracy: (0, 14, 18, 19) and the corrsponding labels are [[1.0], [0.0], [1.0], [0.0]]
Subsets 0: (0, 1, 2, 3) with labels [1.0, 1.0, 1.0, 1.0] has mean distance 0.4797 and accuracy 0.5250 by NN and 0.5250 by RF
Subsets 4582: (9, 12, 13, 17) with labels [1.0, 1.0, 0.0, 1.0] has mean distance 0.4086 and accuracy 1.0000 by NN and 0.6875 by RF
Subsets 3160: (4, 6, 11, 14) with labels [0.0, 0.0, 1.0, 0.0] has mean distance 0.7725 and accuracy 0.0000 by NN and 0.6100 by RF
Subsets 2288: (2, 8

In [24]:
print(len(subset_mean_distances))
print(f"Mean distance: {subset_mean_distances.mean():.4f}")
print(f"Max distance: {subset_mean_distances.max():.4f}")
print(f"Min distance: {subset_mean_distances.min():.4f}")
distance_lu = (1-(-1)) / (2 ** 0.5)  # point at left-upper corner, farthest from the decision boundary
print(f"Distance from decision boundary to left-upper corner: {distance_lu:.4f}")
distance_rb = (-1-1) / (2 ** 0.5)  # point at right-bottomcorner, farthest from the decision boundary
print(f"Distance from decision boundary to right-bottom corner: {distance_rb:.4f}")

print(np.corrcoef(subset_mean_distances, accuracies.numpy()))

4845
Mean distance: 0.5339
Max distance: 0.9904
Min distance: 0.1867
Distance from decision boundary to left-upper corner: 1.4142
Distance from decision boundary to right-bottom corner: -1.4142
[[1.         0.00867237]
 [0.00867237 1.        ]]


In [25]:
#  class balance of each subset
subset_class_balance = []
for subset in all_subsets:
    y_subset = labels[list(subset)].mean().item()
    subset_class_balance.append(y_subset)

subset_class_balance = np.array(subset_class_balance)

plt.figure(figsize=(10, 6))
plt.scatter(subset_class_balance, accuracies.numpy(), alpha=0.1, s=10)
plt.xlabel('Class 1 Proportion in Subset (0=all class0, 0.5=balanced, 1=all class1)')
plt.ylabel('Accuracy')
plt.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Perfect Balance (0.5)')
plt.title(f'Accuracy vs Class Balance (n={unlabeled}, k={label_budget})\n'
          f'cor={np.corrcoef(subset_class_balance, accuracies.numpy())[0,1]:.4f}')
plt.legend()
plt.tight_layout()
plt.savefig('Accuracy_vs_ClassBalance.png')
print('pic saved')

fig, ax = plt.subplots(figsize=(10, 6))

labels_map = {0.0: '0:4', 0.25: '1:3', 0.5: '2:2', 0.75: '3:1', 1.0: '4:0'}
groups = [accuracies.numpy()[np.array(subset_class_balance) == v] for v in [0.0, 0.25, 0.5, 0.75, 1.0]]

ax.boxplot(groups, labels=labels_map.values())
ax.set_xlabel('Label Distribution (class0:class1)')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy Distribution by Label Balance')
plt.tight_layout()
plt.savefig('Accuracy_Boxplot_LabelBalance.png')



pic saved


C:\Users\yeh75\AppData\Local\Temp\ipykernel_3016\157021306.py:26: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(groups, labels=labels_map.values())


In [26]:
# distribution of label ratios in subsets in the two peaks
subset_label_counts = []  #  number of class 1  (0, 1, 2, 3, 4)
for subset in all_subsets:
    y_subset = labels[list(subset)].sum().int().item()  # class 1 的數量
    subset_label_counts.append(y_subset)

subset_label_counts = np.array(subset_label_counts)
accuracies_np = accuracies.numpy()

# find the subsets in the two peaks
peak1_mask = (accuracies_np >= 0.46) & (accuracies_np <= 0.48)
peak2_mask = (accuracies_np >= 0.52) & (accuracies_np <= 0.54)

# distribution of label ratios in subsets in the two peaks
for peak_name, mask in [('Peak 1 (0.46-0.48)', peak1_mask), 
                         ('Peak 2 (0.52-0.54)', peak2_mask)]:
    counts = subset_label_counts[mask]
    print(f'\n{peak_name}，total {mask.sum()} subsets：')
    for i in range(5):
        n = (counts == i).sum()
        print(f'  {i} point(s) of class 1 ({4-i}:{i} → {n} subsets，{n/mask.sum()*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
labels_map = {0: '0:4', 1: '1:3', 2: '2:2', 3: '3:1', 4: '4:0'}
colors = ['#d32f2f', '#f57c00', '#388e3c', '#1976d2', '#7b1fa2']

for ax, (peak_name, mask) in zip(axes, [('Peak 1 (acc: 0.46-0.48)', peak1_mask),
                                         ('Peak 2 (acc: 0.52-0.54)', peak2_mask)]):
    counts = subset_label_counts[mask]
    values = [(counts == i).sum() for i in range(5)]
    percentages = [v / mask.sum() * 100 for v in values]
    
    bars = ax.bar([labels_map[i] for i in range(5)], values,
                   color=colors, edgecolor='black', alpha=0.8)
    
    for bar, val, pct in zip(bars, values, percentages):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{val}\n({pct:.1f}%)',
                    ha='center', va='bottom', fontsize=10)
    
    ax.set_xlabel('Label Distribution (class0:class1)')
    ax.set_ylabel('Number of Subsets')
    ax.set_title(f'{peak_name}\n(Total: {mask.sum()} subsets)')

plt.suptitle('Label Distribution of Subsets in the Two Accuracy Peaks',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('Peak_Label_Distribution.png')
print('pic saved')


Peak 1 (0.46-0.48)，total 754 subsets：
  0 point(s) of class 1 (4:0 → 27 subsets，3.6%)
  1 point(s) of class 1 (3:1 → 184 subsets，24.4%)
  2 point(s) of class 1 (2:2 → 315 subsets，41.8%)
  3 point(s) of class 1 (1:3 → 197 subsets，26.1%)
  4 point(s) of class 1 (0:4 → 31 subsets，4.1%)

Peak 2 (0.52-0.54)，total 742 subsets：
  0 point(s) of class 1 (4:0 → 27 subsets，3.6%)
  1 point(s) of class 1 (3:1 → 164 subsets，22.1%)
  2 point(s) of class 1 (2:2 → 335 subsets，45.1%)
  3 point(s) of class 1 (1:3 → 185 subsets，24.9%)
  4 point(s) of class 1 (0:4 → 31 subsets，4.2%)
pic saved


In [27]:
#   distribution of label ratios in subsets in ranges of top 20% and bottom 20%
subset_label_counts = []  #  number of class 1  (0, 1, 2, 3, 4)
for subset in all_subsets:
    y_subset = labels[list(subset)].sum().int().item()  # class 1 的數量
    subset_label_counts.append(y_subset)

subset_label_counts = np.array(subset_label_counts)
accuracies_np = accuracies.numpy()

# find the subsets in the ranges of top 20% and bottom 20%
top20_mask = (accuracies_np >= 0.80) & (accuracies_np <= 1.00)
btm20_mask = (accuracies_np >= 0.00) & (accuracies_np <= 0.20)

#   distribution of label ratios in subsets in the two peaks
for peak_name, mask in [('Top 20 (0.80-1.00)', top20_mask), 
                         ('Bottom 20 (0.00-0.20)', btm20_mask)]:
    counts = subset_label_counts[mask]
    print(f'\n{peak_name}，total {mask.sum()} subsets：')
    for i in range(5):
        n = (counts == i).sum()
        print(f'  {i} point(s) of class 1 ({4-i}:{i} → {n} subsets，{n/mask.sum()*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
labels_map = {0: '0:4', 1: '1:3', 2: '2:2', 3: '3:1', 4: '4:0'}
colors = ['#d32f2f', '#f57c00', '#388e3c', '#1976d2', '#7b1fa2']

for ax, (peak_name, mask) in zip(axes, [('Top 20 (acc: 0.80-1.00)', top20_mask),
                                         ('Bottom 20 (acc: 0.00-0.20)', btm20_mask)]):
    counts = subset_label_counts[mask]
    values = [(counts == i).sum() for i in range(5)]
    percentages = [v / mask.sum() * 100 for v in values]
    
    bars = ax.bar([labels_map[i] for i in range(5)], values,
                   color=colors, edgecolor='black', alpha=0.8)
    
    for bar, val, pct in zip(bars, values, percentages):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{val}\n({pct:.1f}%)',
                    ha='center', va='bottom', fontsize=10)
    
    ax.set_xlabel('Label Distribution (class0:class1)')
    ax.set_ylabel('Number of Subsets')
    ax.set_title(f'{peak_name}\n(Total: {mask.sum()} subsets)')

plt.suptitle('Label Distribution of Subsets in the top 20% and bottom 20% Accuracy Groups',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('Top20BTM20_Label_Distribution.png')
print('pic saved')


Top 20 (0.80-1.00)，total 280 subsets：
  0 point(s) of class 1 (4:0 → 13 subsets，4.6%)
  1 point(s) of class 1 (3:1 → 67 subsets，23.9%)
  2 point(s) of class 1 (2:2 → 119 subsets，42.5%)
  3 point(s) of class 1 (1:3 → 70 subsets，25.0%)
  4 point(s) of class 1 (0:4 → 11 subsets，3.9%)

Bottom 20 (0.00-0.20)，total 263 subsets：
  0 point(s) of class 1 (4:0 → 11 subsets，4.2%)
  1 point(s) of class 1 (3:1 → 65 subsets，24.7%)
  2 point(s) of class 1 (2:2 → 113 subsets，43.0%)
  3 point(s) of class 1 (1:3 → 64 subsets，24.3%)
  4 point(s) of class 1 (0:4 → 10 subsets，3.8%)
pic saved


In [28]:
peak1_indices = np.where(peak1_mask)[0]
peak2_indices = np.where(peak2_mask)[0]

# 比較前幾個 subset 的點
for i in range(5):
    print(f'Peak1 subset {peak1_indices[i]}: {all_subsets[peak1_indices[i]]}')
    print(f'Peak2 subset {peak2_indices[i]}: {all_subsets[peak2_indices[i]]}')
    print('---')

Peak1 subset 10: (0, 1, 2, 13)
Peak2 subset 0: (0, 1, 2, 3)
---
Peak1 subset 21: (0, 1, 3, 8)
Peak2 subset 8: (0, 1, 2, 11)
---
Peak1 subset 37: (0, 1, 4, 9)
Peak2 subset 14: (0, 1, 2, 17)
---
Peak1 subset 41: (0, 1, 4, 13)
Peak2 subset 19: (0, 1, 3, 6)
---
Peak1 subset 42: (0, 1, 4, 14)
Peak2 subset 32: (0, 1, 3, 19)
---


In [29]:
# 找一個 peak1 和一個 peak2 的 subset
idx1 = peak1_indices[0] # acc ca. 0.47
idx2 = peak2_indices[0] # acc ca. 0.53

with torch.no_grad():
    logits1 = x.unsqueeze(0) @ W[idx1].unsqueeze(0).transpose(-1,-2) + b[idx1].unsqueeze(0).unsqueeze(1)
    logits2 = x.unsqueeze(0) @ W[idx2].unsqueeze(0).transpose(-1,-2) + b[idx2].unsqueeze(0).unsqueeze(1)
    preds1 = (logits1 >= 0).float().squeeze()
    preds2 = (logits2 >= 0).float().squeeze()

print(f'acc1: {accuracies[idx1]:.4f}, acc2: {accuracies[idx2]:.4f}')
print(f'acc1 + acc2 = {accuracies[idx1] + accuracies[idx2]:.4f}')
print(f'NN in Peak 1 predicts {preds1.mean().item():.2%} of 400 points in test set as class1 ')
print(f'NN in Peak 2 predicts {preds2.mean().item():.2%} of 400 points in test set as class1 ')
print(f'How possible are their predictions different: {(preds1 != preds2).float().mean().item():.2%}')
print('=='*30)

acc1: 0.4750, acc2: 0.5250
acc1 + acc2 = 1.0000
NN in Peak 1 predicts 0.00% of 400 points in test set as class1 
NN in Peak 2 predicts 100.00% of 400 points in test set as class1 
How possible are their predictions different: 100.00%


In [30]:
# 找一個 peak1 和一個 peak2 的 subset
for i in range(10):
    idx1 = peak1_indices[i] # acc ca. 0.47
    idx2 = peak2_indices[i] # acc ca. 0.53

    with torch.no_grad():
        logits1 = x.unsqueeze(0) @ W[idx1].unsqueeze(0).transpose(-1,-2) + b[idx1].unsqueeze(0).unsqueeze(1)
        logits2 = x.unsqueeze(0) @ W[idx2].unsqueeze(0).transpose(-1,-2) + b[idx2].unsqueeze(0).unsqueeze(1)
        preds1 = (logits1 >= 0).float().squeeze()
        preds2 = (logits2 >= 0).float().squeeze()
 
    print(f'acc1: {accuracies[idx1]:.4f}, acc2: {accuracies[idx2]:.4f}')
    print(f'acc1 + acc2 = {accuracies[idx1] + accuracies[idx2]:.4f}')
    print(f'NN in Peak 1 predicts {preds1.mean().item():.2%} of 400 points in test set as class1 ')
    print(f'NN in Peak 2 predicts {preds2.mean().item():.2%} of 400 points in test set as class1 ')
    print(f'How possible are their predictions different: {(preds1 != preds2).float().mean().item():.2%}')
    print('=='*30)

acc1: 0.4750, acc2: 0.5250
acc1 + acc2 = 1.0000
NN in Peak 1 predicts 0.00% of 400 points in test set as class1 
NN in Peak 2 predicts 100.00% of 400 points in test set as class1 
How possible are their predictions different: 100.00%
acc1: 0.4750, acc2: 0.5225
acc1 + acc2 = 0.9975
NN in Peak 1 predicts 0.00% of 400 points in test set as class1 
NN in Peak 2 predicts 99.75% of 400 points in test set as class1 
How possible are their predictions different: 99.75%
acc1: 0.4800, acc2: 0.5250
acc1 + acc2 = 1.0050
NN in Peak 1 predicts 0.50% of 400 points in test set as class1 
NN in Peak 2 predicts 100.00% of 400 points in test set as class1 
How possible are their predictions different: 99.50%
acc1: 0.4600, acc2: 0.5250
acc1 + acc2 = 0.9850
NN in Peak 1 predicts 93.50% of 400 points in test set as class1 
NN in Peak 2 predicts 100.00% of 400 points in test set as class1 
How possible are their predictions different: 6.50%
acc1: 0.4800, acc2: 0.5300
acc1 + acc2 = 1.0100
NN in Peak 1 predict

In [31]:
high_acc_indices = torch.where(accuracies >= 0.75)[0]
idx_high = high_acc_indices[0]

with torch.no_grad():
    logits_high = x.unsqueeze(0) @ W[idx_high].unsqueeze(0).transpose(-1,-2) + b[idx_high].unsqueeze(0).unsqueeze(1)
    preds_high = (logits_high >= 0).float().squeeze()

print(f'High acc subset: {all_subsets[idx_high]}')
print(f'Accuracy: {accuracies[idx_high]:.4f}')
print(f'{preds_high.mean().item():.2%} will be predicted as class1 by this high-acc NN')

High acc subset: (0, 1, 2, 15)
Accuracy: 0.8000
32.50% will be predicted as class1 by this high-acc NN


In [32]:
high_acc_indices = torch.where(accuracies >= 0.70)[0]
for i in range(min(10, len(high_acc_indices))):
    idx_high = high_acc_indices[i]

    with torch.no_grad():
        logits_high = x.unsqueeze(0) @ W[idx_high].unsqueeze(0).transpose(-1,-2) + b[idx_high].unsqueeze(0).unsqueeze(1)
        preds_high = (logits_high >= 0).float().squeeze()

    print(f'High acc subset: {all_subsets[idx_high]}')
    print(f'Accuracy: {accuracies[idx_high]:.4f}')
    print(f'{preds_high.mean().item():.2%} will be predicted as class1 by this high-acc NN')

High acc subset: (0, 1, 2, 15)
Accuracy: 0.8000
32.50% will be predicted as class1 by this high-acc NN
High acc subset: (0, 1, 2, 16)
Accuracy: 0.7275
79.75% will be predicted as class1 by this high-acc NN
High acc subset: (0, 1, 3, 9)
Accuracy: 0.7425
26.75% will be predicted as class1 by this high-acc NN
High acc subset: (0, 1, 3, 15)
Accuracy: 0.7850
58.00% will be predicted as class1 by this high-acc NN
High acc subset: (0, 1, 4, 7)
Accuracy: 0.8650
50.00% will be predicted as class1 by this high-acc NN
High acc subset: (0, 1, 5, 15)
Accuracy: 0.7775
37.75% will be predicted as class1 by this high-acc NN
High acc subset: (0, 1, 5, 17)
Accuracy: 0.7750
33.50% will be predicted as class1 by this high-acc NN
High acc subset: (0, 1, 6, 7)
Accuracy: 0.8225
65.75% will be predicted as class1 by this high-acc NN
High acc subset: (0, 1, 8, 12)
Accuracy: 0.7025
47.25% will be predicted as class1 by this high-acc NN
High acc subset: (0, 1, 8, 19)
Accuracy: 0.8075
57.75% will be predicted as 

In [33]:
# min distance to decision boundary
distances = (data[:, 1] - data[:, 0]).abs() / (2 ** 0.5)

subset_min_distances = []
for subset in all_subsets:
    d = distances[list(subset)].min().item()
    subset_min_distances.append(d)

subset_min_distances = np.array(subset_min_distances)

correlation = np.corrcoef(subset_min_distances, accuracies_np)[0, 1]
print(f'Correlation (min distance): {correlation:.4f}')

plt.figure(figsize=(10, 6))
plt.scatter(subset_min_distances, accuracies_np, alpha=0.1, s=10)
plt.xlabel('Min Distance to Decision Boundary')
plt.ylabel('Accuracy')
plt.title(f'Accuracy vs Min Distance (n={unlabeled}, k={label_budget})\ncor={correlation:.4f}')
plt.tight_layout()
plt.savefig('Accuracy_vs_MinDistance.png')

Correlation (min distance): 0.0129


In [34]:
# Convex hull of the 4 points in each subset and whether it crosses the decision boundary
# x1 - x0 > 0 → class 1 
# x1 - x0 < 0 → class 0 

side = data[:, 1] - data[:, 0]  # positive=class1，negative=class0

subset_crosses_boundary = []
for subset in all_subsets:
    sides = side[list(subset)]
    has_positive = (sides > 0).any()
    has_negative = (sides < 0).any()
    crosses = (has_positive & has_negative).item()
    subset_crosses_boundary.append(crosses)

subset_crosses_boundary = np.array(subset_crosses_boundary)
print(f'{subset_crosses_boundary.sum()} subsets cross the decision boundary')
print(f'{(~subset_crosses_boundary).sum()} subsets do not cross the decision boundary')
crosses_acc = accuracies_np[subset_crosses_boundary]
not_crosses_acc = accuracies_np[~subset_crosses_boundary]

print(f'average accuracy of subsets that cross the boundary: {crosses_acc.mean():.4f}, std={crosses_acc.std():.4f}, min={crosses_acc.min():.4f}, max={crosses_acc.max():.4f}')
print(f'average accuracy of subsets that do not cross the boundary: {not_crosses_acc.mean():.4f}, std={not_crosses_acc.std():.4f}, min={not_crosses_acc.min():.4f}, max={not_crosses_acc.max():.4f}')

4425 subsets cross the decision boundary
420 subsets do not cross the decision boundary
average accuracy of subsets that cross the boundary: 0.5016, std=0.1774, min=0.0000, max=1.0000
average accuracy of subsets that do not cross the boundary: 0.5058, std=0.1775, min=0.0600, max=0.9500


In [35]:
# points in the subset are symmetrically distributed around the decision boundary or not
subset_symmetry = []
for subset in all_subsets:
    points = data[list(subset)]
    sides = (points[:, 1] - points[:, 0])
    
    class0_mask = sides < 0
    class1_mask = sides > 0
    
    # ignore subsets that are all on one side (symmetry is not defined for them)
    if class0_mask.sum() == 0 or class1_mask.sum() == 0:
        subset_symmetry.append(np.nan)
        continue
    
    class0_dist = distances[list(subset)][class0_mask].mean()
    class1_dist = distances[list(subset)][class1_mask].mean()
    symmetry = abs(class0_dist - class1_dist).item()
    subset_symmetry.append(symmetry)

subset_symmetry = np.array(subset_symmetry)

# calculate correlation only for subsets that have valid symmetry values (not NaN)
valid_mask = ~np.isnan(subset_symmetry)
correlation = np.corrcoef(subset_symmetry[valid_mask], accuracies_np[valid_mask])[0, 1]
print(f'Correlation (symmetry): {correlation:.4f}')

Correlation (symmetry): -0.0021


In [36]:
# mean pairwise distance of the 4 points in each subset
# calculate mean pairwise distance for each subset
subset_diversities = []
for subset in all_subsets:
    points = data[list(subset)]  # [4, 2]
    pairs = list(combinations(range(4), 2))  # C(4,2) = 6 pairs
    pair_distances = [torch.dist(points[i], points[j]).item() for i, j in pairs]
    subset_diversities.append(np.mean(pair_distances))

subset_diversities = np.array(subset_diversities)

correlation = np.corrcoef(subset_diversities, accuracies_np)[0, 1]
print(f'Correlation (diversity): {correlation:.4f}')

# Visualization
plt.figure(figsize=(10, 6))

plt.scatter(subset_diversities, accuracies_np, alpha=0.1, s=10)
plt.xlabel('Mean Pairwise Distance')
plt.ylabel('Accuracy')
plt.title(f'Accuracy vs mean pairwise distance of the 4 points\ncor={correlation:.4f}')
plt.tight_layout()
plt.savefig('Accuracy_vs_pairwise distance.png')
print('pic saved')

Correlation (diversity): -0.0030
pic saved


In [37]:
'''# test ave. distance of pairwise points within the subset
subset = np.ones((4, 2))  # fill this with the subset coordinates

ss_list = []
for subset in all_subsets:
    subset_points = data[list(subset)].cpu().numpy()  # get the coordinates of the points in the subset
    ss = subset_points[:, np.newaxis, :] - subset_points[np.newaxis, :, :]    # pairwise differences
    print(ss.shape)  # should be (4, 4, 2)
    ss = np.sqrt(np.power(ss, 2).sum(axis=-1))  # pairwise distances
    print(ss.shape)  # should be (4, 4)
    ss_list.append(ss.mean())  # average distance for this subset

correlation = np.corrcoef(ss_list, accuracies_np)[0, 1]
print(f'Correlation): {correlation:.4f}')


# Visualization
plt.figure(figsize=(10, 6))
plt.scatter(ss_list, accuracies_np, alpha=0.1, s=10)
plt.xlabel('Mean Pairwise Distance')
plt.ylabel('Accuracy')
plt.title(f'Accuracy vs mean pairwise distance of the 4 points\ncor={correlation:.4f}')
plt.tight_layout()
plt.savefig('Accuracy_vs_pairwise distance_bb.png')
print('pic saved')'''

"# test ave. distance of pairwise points within the subset\nsubset = np.ones((4, 2))  # fill this with the subset coordinates\n\nss_list = []\nfor subset in all_subsets:\n    subset_points = data[list(subset)].cpu().numpy()  # get the coordinates of the points in the subset\n    ss = subset_points[:, np.newaxis, :] - subset_points[np.newaxis, :, :]    # pairwise differences\n    print(ss.shape)  # should be (4, 4, 2)\n    ss = np.sqrt(np.power(ss, 2).sum(axis=-1))  # pairwise distances\n    print(ss.shape)  # should be (4, 4)\n    ss_list.append(ss.mean())  # average distance for this subset\n\ncorrelation = np.corrcoef(ss_list, accuracies_np)[0, 1]\nprint(f'Correlation): {correlation:.4f}')\n\n\n# Visualization\nplt.figure(figsize=(10, 6))\nplt.scatter(ss_list, accuracies_np, alpha=0.1, s=10)\nplt.xlabel('Mean Pairwise Distance')\nplt.ylabel('Accuracy')\nplt.title(f'Accuracy vs mean pairwise distance of the 4 points\ncor={correlation:.4f}')\nplt.tight_layout()\nplt.savefig('Accuracy_v

In [38]:
# from dustin 
all_ss_values = []

for subset_idx in all_subsets:
    subset_points = data[list(subset_idx)].numpy() 
    
    # matrix of pairwise differences: shape (4, 4, 2)，diff[i,j] = subset_points[i] - subset_points[j]
    diff = subset_points[:, np.newaxis, :] - subset_points[np.newaxis, :, :]    
    #print(diff.shape)
    dist_matrix = np.sqrt(np.power(diff, 2).sum(axis=-1))
    #print(dist_matrix.shape)
    
    # take the upper triangle of the distance matrix (excluding the diagonal) to get the 6 pairwise distances
    mask = np.triu_indices(4, k=1)
    mean_dist = dist_matrix[mask].mean()
    all_ss_values.append(mean_dist)

all_ss_values = np.array(all_ss_values)

correlation = np.corrcoef(all_ss_values, accuracies_np)[0, 1]
print(f'Correlation (Diversity): {correlation:.4f}')

# Visualization
plt.figure(figsize=(10, 6))
plt.scatter(all_ss_values, accuracies_np, alpha=0.1, s=10)
plt.xlabel('Mean Pairwise Distance')
plt.ylabel('Accuracy')
plt.title(f'Accuracy vs mean pairwise distance of the 4 points\ncor={correlation:.4f}')
plt.tight_layout()
plt.savefig('Accuracy_vs_pairwise distance+bbb.png')
print('pic saved')

Correlation (Diversity): -0.0030
pic saved


In [ ]:
'''from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

n_subsets = len(accuracies_np)
sorted_indices = np.argsort(accuracies_np)

bottom_10 = sorted_indices[:int(n_subsets * 0.1)]
middle_10 = sorted_indices[int(n_subsets * 0.45):int(n_subsets * 0.55)]
top_10    = sorted_indices[int(n_subsets * 0.9):]

data_np = data.numpy()
labels_np = labels.flatten().numpy()

# vmax should be the same across all 3 plots to make the color intensity comparable, so we calculate it globally
all_counts = []
for group_indices in [bottom_10, middle_10, top_10]:
    point_counts = np.zeros(len(data_np))
    for idx in group_indices:
        for p_idx in all_subsets[idx]:
            point_counts[p_idx] += 1
    all_counts.append(point_counts)
vmax_global = max([c.max() for c in all_counts])
norm = Normalize(vmin=0, vmax=vmax_global)

fig, axes = plt.subplots(1, 3, figsize=(28, 10))
fig.subplots_adjust(right=0.90) 

for ax, (group_name, group_indices), point_counts in zip(axes, [
    ('Bottom 10%', bottom_10),
    ('Middle 10%', middle_10),
    ('Top 10%',    top_10)
], all_counts):

    ax.plot([-1, 1], [-1, 1], color='gray', linestyle='--',
            alpha=0.5, label='Decision Boundary', zorder=1)

    for class_val, marker, cmap, label in [
        (0, 'o', 'Blues',  'Class 0'),
        (1, 'o', 'Blues', 'Class 1')
    ]:
        mask = labels_np == class_val
        ax.scatter(
            data_np[mask, 0], data_np[mask, 1],
            c=point_counts[mask],
            cmap=cmap,
            marker=marker,
            s=150,
            edgecolors='black',
            linewidths=0.5,
            norm=norm,    
            label=label,
            zorder=2
        )

    acc_mean = accuracies_np[group_indices].mean()
    acc_min  = accuracies_np[group_indices].min()
    acc_max  = accuracies_np[group_indices].max()

    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_xlabel('x0')
    ax.set_ylabel('x1')
    ax.set_title(f'{group_name} ({len(group_indices)} subsets)\n'
                 f'acc: {acc_min:.3f} - {acc_max:.3f}, mean: {acc_mean:.3f}')
    ax.legend(loc='upper left', fontsize=12, markerscale=1.2, framealpha=0.5)

#  colorbars
#cax_red  = fig.add_axes([0.87, 0.15, 0.012, 0.65])
cax_blue = fig.add_axes([0.91, 0.15, 0.012, 0.65])

#fig.colorbar(ScalarMappable(norm=norm, cmap='Reds'),  cax=cax_red,  label='Class 0 Count')
fig.colorbar(ScalarMappable(norm=norm, cmap='Blues'), cax=cax_blue, label='Class 1 Count')

plt.suptitle(f'Point Distribution: Top / Middle / Bottom 10% Subsets\n'
             f'(n={unlabeled}, k={label_budget})',
             fontsize=14, fontweight='bold')

plt.savefig(f'Point_Distribution_Top_Middle_Bottom_10pct_{len(all_subsets)}.png',
            bbox_inches='tight')
print('pic saved')'''

pic saved


In [61]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

n_subsets = len(accuracies_np)
sorted_indices = np.argsort(accuracies_np)

bottom_10 = sorted_indices[:int(n_subsets * 0.1)]
middle_10 = sorted_indices[int(n_subsets * 0.45):int(n_subsets * 0.55)]
top_10    = sorted_indices[int(n_subsets * 0.9):]

data_np = data.numpy()
labels_np = labels.flatten().numpy()

all_counts = []
for group_indices in [bottom_10, middle_10, top_10]:
    point_counts = np.zeros(len(data_np))
    for idx in group_indices:
        for p_idx in all_subsets[idx]:
            point_counts[p_idx] += 1
    all_counts.append(point_counts)

vmax_global = max([c.max() for c in all_counts])

norm = Normalize(vmin=0, vmax=100)

fig, axes = plt.subplots(1, 3, figsize=(28, 10))
fig.subplots_adjust(right=0.90)

for ax, (group_name, group_indices), point_counts in zip(axes, [
    ('Bottom 10%', bottom_10),
    ('Middle 10%', middle_10),
    ('Top 10%',    top_10)
], all_counts):

    ax.plot([-1, 1], [-1, 1], color='gray', linestyle='--',
            alpha=0.5, label='Decision Boundary', zorder=1)

    point_pct = point_counts / vmax_global * 100

    ax.scatter(
        data_np[:, 0], data_np[:, 1],
        c=point_pct,
        cmap='GnBu',
        marker='o',
        s=1200,
        edgecolors='black',
        linewidths=0.5,
        norm=norm,
        label='Data Points',
        zorder=2
    )

    acc_mean = accuracies_np[group_indices].mean()
    acc_min  = accuracies_np[group_indices].min()
    acc_max  = accuracies_np[group_indices].max()

    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_xlabel('x0')
    ax.set_ylabel('x1')
    ax.set_title(f'{group_name} ({len(group_indices)} subsets)\n'
                 f'acc: {acc_min:.3f} - {acc_max:.3f}, mean: {acc_mean:.3f}', fontsize=18)
    ax.legend(loc='upper left', fontsize=12, markerscale=0.8, framealpha=0.5)

cax_gnbu = fig.add_axes([0.91, 0.15, 0.012, 0.65])
fig.colorbar(ScalarMappable(norm=norm, cmap='GnBu'),
             cax=cax_gnbu, label='Selection Count (%)')

plt.suptitle(f'Point Distribution: Top / Middle / Bottom 10% Subsets\n'
             f'(n={unlabeled}, k={label_budget})',
             fontsize=14, fontweight='bold')

plt.savefig(f'Point_Distribution_Top_Middle_Bottom_10pct_{len(all_subsets)}.png',
            bbox_inches='tight')
print('pic saved')

pic saved
